<a href="https://colab.research.google.com/github/zion2010p/ai/blob/main/ai_%EB%82%9C%EC%88%98_%EC%83%9D%EC%84%B1%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-genai

In [ ]:
"""
AI 기반 동적 난수 생성기 (Google Colab 호환)
=============================================
Gemini AI가 매번 새로운 난수 생성 알고리즘을 실시간으로 창작합니다.

API 키 발급: https://aistudio.google.com/apikey (무료)
"""

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📌 여기에 API 키 입력
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
API_KEY = ""  # ← Gemini API 키

import os, re, time, math, hashlib
from datetime import datetime

if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GEMINI_API_KEY")
        print("✅ Colab Secrets에서 API 키를 가져왔습니다.")
    except Exception:
        API_KEY = os.environ.get("GEMINI_API_KEY", "")

if not API_KEY:
    print("⚠️  API 키가 필요합니다!")
    print("   → API_KEY 변수에 직접 입력하거나")
    print("   → Colab 🔑 시크릿에 GEMINI_API_KEY 추가")
    print("   → 발급: https://aistudio.google.com/apikey")

from google import genai

# 최신 모델 우선순위 (2025~2026 현재 사용 가능한 모델)
MODELS = [
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",
]

PROMPT_TEMPLATE = """\
Create a unique random number generation algorithm in Python.

Rules:
1. Return one integer between {min_val} and {max_val} from a function called `generate_random()`
2. Do NOT use the `random` module. You CAN use: time, math, os, hashlib, struct
3. Use a creative mathematical/bitwise/entropy approach
4. First line must be a comment: # [Name]: [Description]

Previously used approaches (use something DIFFERENT):
{history}

Respond with ONLY:
```python
# [Name]: [Description]
def generate_random():
    ...
    return value
```"""


class AIRandomGenerator:
    """AI가 매번 새로운 난수 알고리즘을 생성"""

    def __init__(self, api_key: str):
        self.client = genai.Client(api_key=api_key)
        self.algorithm_history = []
        self._working_model = None  # 작동 확인된 모델 캐시

    def _find_working_model(self):
        """사용 가능한 모델을 찾아 캐시합니다."""
        if self._working_model:
            return self._working_model

        print("   🔍 사용 가능한 모델 확인 중...")
        for model in MODELS:
            try:
                resp = self.client.models.generate_content(
                    model=model, contents="Say OK"
                )
                if resp.text:
                    self._working_model = model
                    print(f"   ✅ 모델 선택: {model}")
                    return model
            except Exception as e:
                err = str(e)
                if "404" in err or "NOT_FOUND" in err:
                    continue  # 모델 없음 → 다음
                if "429" in err or "RESOURCE_EXHAUSTED" in err:
                    # 모델은 존재하지만 할당량 초과
                    wait = self._parse_wait(err)
                    print(f"   ⏳ {model} 할당량 초과, {wait}초 대기...")
                    time.sleep(wait)
                    self._working_model = model
                    return model
                continue

        raise RuntimeError(
            "사용 가능한 모델이 없습니다.\n"
            "→ API 키를 확인하거나 잠시 후 다시 시도해주세요."
        )

    def _call_ai(self, prompt: str) -> str:
        """AI 호출. 429시 자동 대기 후 재시도 (최대 3회)"""
        model = self._find_working_model()

        for attempt in range(3):
            try:
                resp = self.client.models.generate_content(
                    model=model, contents=prompt
                )
                return resp.text
            except Exception as e:
                err = str(e)
                if "429" in err or "RESOURCE_EXHAUSTED" in err:
                    wait = self._parse_wait(err)
                    print(f"   ⏳ 할당량 초과, {wait}초 대기 중... ({attempt+1}/3)")
                    time.sleep(wait)
                elif "404" in err or "NOT_FOUND" in err:
                    self._working_model = None  # 캐시 초기화
                    model = self._find_working_model()
                else:
                    raise

        raise RuntimeError("최대 재시도 횟수 초과. 잠시 후 다시 시도해주세요.")

    def _parse_wait(self, err: str) -> float:
        """에러에서 대기 시간 추출"""
        m = re.search(r"retry in (\d+\.?\d*)", err, re.IGNORECASE)
        return min(float(m.group(1)) + 2, 120) if m else 15

    def generate(self, min_val=0, max_val=100):
        """
        새로운 알고리즘으로 난수 생성

        예시:
            gen.generate()          # 0~100
            gen.generate(1, 1000)   # 1~1000
        """
        history = "\n".join(
            f"  - {h['algorithm_name']}" for h in self.algorithm_history
        ) or "  (none yet)"

        prompt = PROMPT_TEMPLATE.format(
            min_val=min_val, max_val=max_val, history=history
        )

        raw = self._call_ai(prompt)
        code = self._extract_code(raw)
        if not code:
            raise RuntimeError(f"코드 추출 실패\n{raw}")

        name = self._get_name(code)
        result = self._run(code, min_val, max_val)

        record = {
            "algorithm_name": name,
            "algorithm_code": code,
            "result": result,
            "timestamp": datetime.now().strftime("%H:%M:%S"),
            "attempt": len(self.algorithm_history) + 1,
        }
        self.algorithm_history.append(record)
        self._show(record)
        return record

    def _extract_code(self, text):
        for pat in [r"```python\s*\n(.*?)```", r"```\s*\n(.*?)```"]:
            m = re.search(pat, text, re.DOTALL)
            if m: return m.group(1).strip()
        if "def generate_random" in text:
            lines, out, started = text.split("\n"), [], False
            for ln in lines:
                if ln.strip().startswith("#") and not started: out.append(ln)
                if "def generate_random" in ln: started = True
                if started:
                    out.append(ln)
                    if ln.strip().startswith("return "): break
            return "\n".join(out) if out else None
        return None

    def _get_name(self, code):
        first = code.split("\n")[0].strip()
        return first.lstrip("# ").strip() if first.startswith("#") else "Unknown"

    def _run(self, code, mn, mx):
        import struct, sys
        builtins = {
            k: v for k, v in {
                "abs": abs, "int": int, "float": float, "len": len,
                "range": range, "list": list, "tuple": tuple, "str": str,
                "hex": hex, "bin": bin, "ord": ord, "chr": chr,
                "max": max, "min": min, "sum": sum, "pow": pow,
                "round": round, "enumerate": enumerate, "zip": zip,
                "map": map, "filter": filter, "sorted": sorted,
                "reversed": reversed, "bool": bool, "bytes": bytes,
                "bytearray": bytearray, "isinstance": isinstance,
                "type": type, "print": print, "__import__": __import__,
                "True": True, "False": False, "None": None,
            }.items()
        }
        ns = {
            "__builtins__": builtins,
            "time": time, "math": math, "os": os,
            "hashlib": hashlib, "struct": struct, "sys": sys,
        }
        exec(code, ns)
        result = int(ns["generate_random"]())
        if not (mn <= result <= mx):
            result = mn + (abs(result) % (mx - mn + 1))
        return result

    def _show(self, r):
        print()
        print("╔" + "═" * 58 + "╗")
        print(f"║  🎲 난수: {r['result']:<47}║")
        print(f"║  📝 알고리즘: {r['algorithm_name'][:42]:<43}║")
        print(f"║  🔢 #{r['attempt']} | ⏰ {r['timestamp']:<40}║")
        print("╠" + "═" * 58 + "╣")
        for ln in r["algorithm_code"].split("\n"):
            print(f"║  {ln[:56]:<56}║")
        print("╚" + "═" * 58 + "╝")

    def history(self):
        if not self.algorithm_history:
            print("아직 기록이 없습니다."); return
        print("\n📊 생성 기록")
        print("─" * 50)
        for r in self.algorithm_history:
            print(f"  #{r['attempt']} [{r['timestamp']}] {r['algorithm_name'][:35]} → {r['result']}")
        print("─" * 50)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 초기화
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
if API_KEY:
    gen = AIRandomGenerator(API_KEY)
    print("╔" + "═" * 58 + "╗")
    print("║     🤖 AI 동적 난수 생성기 v3.0                         ║")
    print("╚" + "═" * 58 + "╝")
    print("  gen.generate()        → 0~100 난수")
    print("  gen.generate(1,1000)  → 1~1000 난수")
    print("  gen.history()         → 기록 보기\n")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 로컬 대화형 모드
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
if __name__ == "__main__":
    if not API_KEY:
        API_KEY = input("🔑 API 키: ").strip()
        if not API_KEY: print("❌ 종료"); exit()
        gen = AIRandomGenerator(API_KEY)
        print("🤖 AI 동적 난수 생성기 v3.0\n")

    print("[Enter]=생성  '1 1000'=범위  h=기록  q=종료")

    while True:
        try:
            ui = input("\n🎯 > ").strip().lower()
        except (EOFError, KeyboardInterrupt):
            break

        if ui in ("q","quit","exit"): gen.history(); break
        if ui in ("h","history"): gen.history(); continue

        mn, mx = 0, 100
        if ui:
            p = ui.split()
            if len(p)==2:
                try: mn,mx = int(p[0]),int(p[1])
                except: print("⚠️ 예: 1 1000"); continue
            else:
                try: mx = int(ui)
                except: print("⚠️ 명령 오류"); continue

        print(f"\n🤖 새 알고리즘 생성 중... ({mn}~{mx})")
        try:
            gen.generate(mn, mx)
        except Exception as e:
            print(f"❌ {e}")
